# 05. Финальная E2E-оценка

Проверяется полный каскад: Gunduz image → YOLO11 → grayscale crop → DINOv2 → FAISS top-k. Каталог результата защищён от повторного теста.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, yaml
import pandas as pd

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None
os.chdir(PROJECT_ROOT)
CONFIG = PROJECT_ROOT / 'configs/inference.yaml'
RUN_FROZEN_E2E_TEST = False
ENABLE_CLEARML = False

## Состав системы и frozen benchmark

In [ ]:
from core.config_loader import load_config
config = load_config(CONFIG)
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
required = {
    'YOLO': PROJECT_ROOT / config['paths']['detector_weights'],
    'DINOv2': PROJECT_ROOT / config['paths']['classifier_checkpoint'],
    'genus FAISS': PROJECT_ROOT / config['paths']['genus_index'],
    'species FAISS': PROJECT_ROOT / config['paths']['species_index'],
    'ground truth': PROJECT_ROOT / config['paths']['ground_truth'],
}
for name, path in required.items():
    print(f'{name}: {path} | exists={path.is_file()}')
ground_truth_path = required['ground truth']
if ground_truth_path.is_file():
    ground_truth = pd.read_csv(ground_truth_path)
    print('GT rows:', len(ground_truth))
    if 'source_cohort' in ground_truth.columns: display(ground_truth['source_cohort'].value_counts())

## Однократный запуск

После просмотра frozen test не меняйте модель или параметры на основании этих результатов. Для нового эксперимента задайте другой output directory.

In [ ]:
command = [sys.executable, '-m', 'scripts.run_test_supermodel', '--config', str(CONFIG)]
if ENABLE_CLEARML:
    command += ['--set', 'clearml.enabled=true']
output = PROJECT_ROOT / config['paths']['output_dir']
if RUN_FROZEN_E2E_TEST:
    missing = [name for name, path in required.items() if not path.is_file()]
    assert not missing, f'Missing artifacts: {missing}'
    assert not output.exists(), f'Frozen test уже запускался: {output}'
    subprocess.run(command, check=True)
else:
    print('Frozen E2E test skipped:', ' '.join(map(str, command)))

## Метрики и таблицы

In [ ]:
metrics_json = output / 'metrics.json'
if metrics_json.is_file():
    print(json.dumps(json.loads(metrics_json.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
for path in sorted(output.rglob('*.csv')) if output.exists() else []:
    print(path.relative_to(PROJECT_ROOT), 'rows=', len(pd.read_csv(path)))
if not output.exists():
    print('Reports появятся после E2E test')